# Procurement risk models (Chapter 3, 3.5.7)

Trains the three risk models of the AI and Blockchain-Based Transparent Government Procurement System for Zambia:

- **supervised**: gradient boosting on Phase 1 labels (rule labels on real ZPPA tenders plus injected synthetic patterns)
- **unsupervised**: Isolation Forest on the real ZPPA tenders
- **graph-based**: the bidder co-participation network and its tight groups ("links between competing bidders")

Data: public ZPPA OCDS data from the Open Contracting Partnership data registry (CC BY 4.0) and synthetic data only.

This notebook runs the same scripts as the repository, with fixed seeds, so its results should match the committed
`ai/models/metrics.json`. *Runtime: about 5 minutes on the free CPU runtime. No GPU needed.*

In [ ]:
!git clone --depth 1 https://github.com/Zahn1/ai-blockchain-procurement-zambia.git repo
%cd repo
!pip install -q -r ai/requirements.txt

If Colab asks you to restart the runtime after the install (because of the numpy version), restart it and run the next cell.

In [ ]:
%cd /content/repo
!mkdir -p ai/data/raw
!curl -sSL -o ai/data/raw/zppa-full.jsonl.gz "https://data.open-contracting.org/en/publication/3/download?name=full.jsonl.gz"
!ls -lh ai/data/raw

## 1. Rebuild the dataset and the synthetic data, and check they match the committed files

In [ ]:
!cp ai/data/processed/synthetic_tenders.csv /tmp/committed_tenders.csv
!cp ai/data/processed/synthetic_contracts.csv /tmp/committed_contracts.csv
!python ai/scripts/build_dataset.py ai/data/raw/zppa-full.jsonl.gz ai/data/processed/tenders.csv
!python ai/scripts/synth.py ai/data/processed/tenders.csv ai/data/processed
!python ai/scripts/build_bid_network.py ai/data/raw/zppa-full.jsonl.gz ai/data/processed/bids.csv
!cmp ai/data/processed/synthetic_tenders.csv /tmp/committed_tenders.csv && echo "synthetic tenders: identical"
!cmp ai/data/processed/synthetic_contracts.csv /tmp/committed_contracts.csv && echo "synthetic contracts: identical"

The OCDS registry updates the ZPPA data monthly. If the download is newer than the one used in the repository,
the synthetic files and the results will differ slightly; the repository's run used the file retrieved on 10 October 2026.

## 2. Train and evaluate

In [ ]:
!cp ai/models/metrics.json /tmp/committed_metrics.json
!python ai/risk/train.py

## 3. Results, compared with the committed run

In [ ]:
import json
new = json.load(open('ai/models/metrics.json'))
old = json.load(open('/tmp/committed_metrics.json'))
for k in ['roc_auc', 'recall_at_threshold', 'false_positive_rate', 'real_alerts']:
    print(f"{k:22s} Colab {new['tender'][k]:.4f}   committed {old['tender'][k]:.4f}")
print('graph cartel in top 5%:', new['graph']['cartel_in_top5pct'], '| leakage AUC:', round(new['leakage_auc'], 3))

In [ ]:
from IPython.display import Markdown
Markdown(open('docs/phase4/model-evaluation.md').read())

## 4. Try the explanations

In [ ]:
import sys
sys.path.insert(0, 'ai')
from risk.model import RiskModels
m = RiskModels('ai/models')
r = m.score_tender({'est_value': 195000, 'n_bidders': 1, 'period_days': 1, 'end_weekday': 5, 'award_value': 195000,
                    'buyer_n_awards': 40, 'supplier_wins': 12, 'buyer_supplier_share': 0.7, 'pair_awards_same_month': 6,
                    'method': 'limited', 'category': 'goods'}, explain=True)
print('score', r['score'], 'alert', r['alert'])
for x in r['reasons']: print(' -', x['text'])

## 5. Download the trained models (to place in `ai/models/`)

In [ ]:
!cd ai && zip -q -r /content/risk-models.zip models
from google.colab import files
files.download('/content/risk-models.zip')